<div align="center">

# Predicción de tasas de mortalidad y natalidad a partir de indicadores socioeconómicos

**Proyecto Integrador**

Universidad Internacional del Ecuador — UIDE

---

**Notebook 05 — Optimización de hiperparámetros y evaluación avanzada**

---

Equipo: Guillermo Paredes · Donato Oña · Mateo Villacreses

Junio 2026

</div>

## Introducción y objetivos

En el notebook 03 se compararon cuatro modelos base (Ridge, Random Forest, XGBoost y LightGBM) bajo validación cruzada grupal (`GroupKFold`), confirmando que los modelos de árboles superan al lineal en la predicción de la mortalidad. Este notebook profundiza sobre el modelo ganador para `death_rate` (**XGBoost**) con cuatro objetivos:

1. **Optimización rigurosa de hiperparámetros** con `GridSearchCV`, respetando `GroupKFold` agrupado por país para evitar fuga de información.
2. **Evaluación antes/después** de la optimización y visualización de la morfología de la mortalidad (real vs. predicho).
3. **Interpretabilidad con SHAP** (Shapley Additive exPlanations).
4. **Diagnóstico estadístico de residuales** del modelo optimizado.

> **Notebook autocontenido.** A diferencia de su versión previa, este notebook **carga el dataset procesado del notebook 02** (`data/processed/dataset_modelado.parquet`) y define su propio entorno de modelado, por lo que no depende de ejecutar antes el notebook 03 en el mismo kernel. Requiere haber ejecutado los notebooks 01 y 02 (que regeneran `data/`).

## 0. Configuración y carga del dataset

Se reconstruye el entorno mínimo de modelado (idéntico al del notebook 03): carga del dataset procesado, definición de `X`, los targets, `groups`, el esquema `GroupKFold` y la fábrica de modelos `construir_modelos()`.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Carpeta para guardar figuras (dentro de data/, no versionada)
FIG_DIR = '../data/figuras'
os.makedirs(FIG_DIR, exist_ok=True)

# --- Carga del dataset procesado (salida del notebook 02) ---
PROCESSED_PATH = '../data/processed/dataset_modelado.parquet'
assert os.path.exists(PROCESSED_PATH), (
    "Falta data/processed/dataset_modelado.parquet. Ejecuta los notebooks 01 y 02 (Run All) antes."
)
df = pd.read_parquet(PROCESSED_PATH)
if 'region_' in df.columns:
    df = df.drop(columns='region_')

META_COLS = ['country_code', 'country_name', 'region', 'income_level', 'year']
TARGETS = ['death_rate', 'birth_rate']
FEATURES = [c for c in df.columns if c not in META_COLS + TARGETS]

X = df[FEATURES].values
y_death = df['death_rate'].values
y_birth = df['birth_rate'].values
groups = df['country_code'].values

cv = GroupKFold(n_splits=5)

def construir_modelos(random_state=RANDOM_STATE):
    return {
        'Ridge': Pipeline([
            ('scaler', StandardScaler()),
            ('reg', Ridge(alpha=1.0, random_state=random_state)),
        ]),
        'RandomForest': RandomForestRegressor(
            n_estimators=300, max_depth=None, min_samples_leaf=2,
            random_state=random_state, n_jobs=-1,
        ),
        'XGBoost': xgb.XGBRegressor(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            random_state=random_state, n_jobs=-1, verbosity=0,
        ),
        'LightGBM': lgb.LGBMRegressor(
            n_estimators=400, max_depth=-1, num_leaves=31, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            random_state=random_state, n_jobs=-1, verbose=-1,
        ),
    }

print(f"Dataset: {df.shape} | features: {len(FEATURES)} | países: {df['country_code'].nunique()}")
print("Setup listo: X, y_death, y_birth, groups, cv, construir_modelos(), FEATURES")

## 1. Optimización del modelo ganador mediante GridSearchCV con validación grupal

Se ajusta finamente **XGBoost** (mejor modelo para `death_rate`) con `GridSearchCV`. Los hiperparámetros controlan la complejidad de los árboles y el ritmo de aprendizaje:

- `learning_rate`: escala la contribución de cada árbol.
- `max_depth`: profundidad máxima (interacciones vs. sobreajuste).
- `subsample` y `colsample_bytree`: muestreo de filas y columnas (regularización por bagging).

**Prevención de fuga de datos:** como el dataset es panel, se pasa el objeto `cv` (`GroupKFold`) y `groups=groups` al `.fit()`, de modo que los pliegues de validación se componen de países nunca vistos en entrenamiento.

In [ ]:
from sklearn.model_selection import GridSearchCV

print("=== OPTIMIZACIÓN DE HIPERPARÁMETROS (XGBoost, death_rate) ===")

base_xgb = construir_modelos()['XGBoost']

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 6, 8],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
}
n_combos = (len(param_grid['learning_rate']) * len(param_grid['max_depth'])
            * len(param_grid['subsample']) * len(param_grid['colsample_bytree']))
print(f"Combinaciones en la grilla: {n_combos} | pliegues GroupKFold: {cv.n_splits}")

grid_search = GridSearchCV(
    estimator=base_xgb, param_grid=param_grid, cv=cv,
    scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1, refit=True,
)
grid_search.fit(X, y_death, groups=groups)

print("\n✅ Búsqueda completada. Mejores hiperparámetros:")
for param, val in grid_search.best_params_.items():
    print(f"  - {param}: {val}")
print(f"Mejor RMSE de validación: {-grid_search.best_score_:.4f}")

## 2. Desempeño y morfología de la mortalidad (real vs. predicho)

Se comparan, mediante predicciones fuera de pliegue (`cross_val_predict`), la línea base (XGBoost por defecto) y el modelo optimizado, usando **RMSE**, **MAE** y **R²**. El gráfico de dispersión real vs. predicho, coloreado por región, permite detectar sesgos geográficos sistemáticos.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Línea base (por defecto) vs modelo optimizado, ambos OOF
y_pred_base = cross_val_predict(base_xgb, X, y_death, groups=groups, cv=cv, n_jobs=-1)
rmse_base = np.sqrt(mean_squared_error(y_death, y_pred_base))
mae_base = mean_absolute_error(y_death, y_pred_base)
r2_base = r2_score(y_death, y_pred_base)

best_xgb = grid_search.best_estimator_
y_pred_opt = cross_val_predict(best_xgb, X, y_death, groups=groups, cv=cv, n_jobs=-1)
rmse_opt = np.sqrt(mean_squared_error(y_death, y_pred_opt))
mae_opt = mean_absolute_error(y_death, y_pred_opt)
r2_opt = r2_score(y_death, y_pred_opt)

print("=== DESEMPEÑO (death_rate) ===")
print(f"Base (default) -> RMSE: {rmse_base:.4f} | MAE: {mae_base:.4f} | R²: {r2_base:.4f}")
print(f"Optimizado     -> RMSE: {rmse_opt:.4f} | MAE: {mae_opt:.4f} | R²: {r2_opt:.4f}")
if r2_base != 0:
    print(f"Mejora relativa R²: {((r2_opt - r2_base) / abs(r2_base)) * 100:+.2f}%")

# Morfología real vs predicho, coloreada por región
plt.figure(figsize=(10, 8))
region_codes = df['region'].astype(str).values
for reg in np.unique(region_codes):
    idx = region_codes == reg
    plt.scatter(y_death[idx], y_pred_opt[idx], label=reg, alpha=0.6, s=30, edgecolors='none')
lims = [min(y_death.min(), y_pred_opt.min()) - 1, max(y_death.max(), y_pred_opt.max()) + 1]
plt.plot(lims, lims, 'r--', lw=2, label='y = ŷ')
plt.xlim(lims); plt.ylim(lims)
plt.xlabel('Mortalidad real (por 1.000 hab.)')
plt.ylabel('Mortalidad predicha (por 1.000 hab.)')
plt.title('Morfología de la mortalidad: real vs. predicho (XGBoost optimizado)')
plt.legend(title='Región', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
stats_text = f"R² = {r2_opt:.4f}\nRMSE = {rmse_opt:.4f}\nMAE = {mae_opt:.4f}"
plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, va='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/05_morfologia_mortalidad.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Interpretabilidad con SHAP

**SHAP** (Shapley Additive exPlanations), basado en teoría de juegos cooperativos, calcula el aporte marginal de cada feature a la desviación de la predicción respecto a la media. A diferencia de las importancias por ganancia/impureza, satisface **aditividad** y **consistencia**.

En el `summary_plot`: el eje Y ordena los features por importancia global, el eje X muestra el impacto SHAP sobre la mortalidad predicha y el color codifica el valor del feature (rojo alto, azul bajo).

In [ ]:
import shap

print("=== VALORES SHAP (XGBoost optimizado) ===")
# Ajustar el mejor modelo sobre todo el dataset para explicar
best_xgb.fit(X, y_death)
X_df = pd.DataFrame(X, columns=FEATURES)

try:
    explainer = shap.TreeExplainer(best_xgb)
    shap_values = explainer.shap_values(X_df)
except Exception as e:
    print(f"Aviso: fallback a Explainer general por: {e}")
    explainer = shap.Explainer(best_xgb, X_df)
    shap_values = explainer(X_df).values

plt.figure()
shap.summary_plot(shap_values, X_df, show=False)
plt.title('SHAP — impacto de los indicadores en la mortalidad predicha', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/05_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Diagnóstico estadístico de residuales

Sobre los residuales OOF del modelo optimizado ($e = y - \hat{y}$) se examinan: **media** (sesgo global), **desviación estándar** (precisión), **asimetría** (subestimación de extremos) y **curtosis** (colas pesadas / errores anómalos).

In [ ]:
from scipy.stats import skew, kurtosis

residuales = y_death - y_pred_opt
mu_error = float(np.mean(residuales))
std_error = float(np.std(residuales))
asimetria = float(skew(residuales))
curtosis_exceso = float(kurtosis(residuales))

print("=== ANÁLISIS DE RESIDUALES (death_rate, OOF) ===")
print(f"Media (sesgo)        : {mu_error:+.4f}")
print(f"Desviación estándar  : {std_error:.4f}")
print(f"Asimetría            : {asimetria:.4f}")
print(f"Curtosis en exceso   : {curtosis_exceso:.4f}")

plt.figure(figsize=(10, 6))
sns.histplot(residuales, kde=True, color='#8e44ad', edgecolor='white', bins=50)
plt.axvline(0, color='#e74c3c', linestyle='--', lw=2, label='Sin sesgo (e = 0)')
plt.axvline(mu_error, color='#2ecc71', linestyle=':', lw=2, label=f'Media: {mu_error:+.3f}')
plt.title('Distribución de residuales de mortalidad (modelo optimizado, OOF)')
plt.xlabel('Residual (real - predicho)')
plt.ylabel('Frecuencia / densidad')
diag = f"mu = {mu_error:.4f}\nsigma = {std_error:.4f}\nasimetría = {asimetria:.4f}\ncurtosis = {curtosis_exceso:.4f}"
plt.gca().text(0.05, 0.95, diag, transform=plt.gca().transAxes, va='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/05_residuales_optimizados.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Conclusiones

- La optimización de hiperparámetros con `GridSearchCV` + `GroupKFold` ajusta XGBoost para `death_rate` sin fuga de información; la mejora sobre la línea base suele ser modesta, coherente con el techo estructural de la mortalidad discutido en el notebook 03.
- El análisis SHAP confirma de forma consistente los determinantes dominantes (estructura etaria y desarrollo económico), aportando interpretabilidad local y global.
- El diagnóstico de residuales permite cuantificar sesgo, dispersión y asimetría, identificando si el modelo subestima sistemáticamente las tasas extremas.
- Las figuras se guardan en `data/figuras/` (no versionada). Los resultados deben leerse junto con las salvedades metodológicas del notebook 04 (alcance del Gini, *ground truth*, outliers por tipo).